In [ ]:
import pandas as pd
import commonlib.prep_export_data as prep_export_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as config

In [ ]:
c = config.Config()
print(c.TERRAFORM_LOG_PATH)
parsed_records=prep_export_data.read_json_from_file(c.TERRAFORM_LOG_PATH)
normalized_records=prep_export_data.normalize_records(parsed_records)
df = pd.json_normalize(normalized_records)

# df_export_start = df[df['type'] == 'export_start']
# df_export_start = df_export_start.rename(columns={'timestamp': 'start_timestamp'})

# df_export_end = df[df['type'] == 'export_end']
# df_export_end = df_export_end.rename(columns={'timestamp': 'end_timestamp'})


# df_merged_refresh = pd.merge(df_export_start[['resource_id', 
# 'start_timestamp','resource','resource_type','resource_label']],
#                              df_export_end[['resource_id', 'end_timestamp']], 
#                              on='resource_id')

# df_merged_refresh['start_datetime'] = pd.to_datetime(df_merged_refresh['start_timestamp'])
# df_merged_refresh['end_datetime'] = pd.to_datetime(df_merged_refresh['end_timestamp'])

# # calculate time difference in minutes
# df_merged_refresh['time_diff_minutes'] = (df_merged_refresh['end_datetime'] - df_merged_refresh['start_datetime']).dt.total_seconds() / 60

# --- Handle multiple exports per resource_id properly ---

# number each start and end event in order per resource_id
starts = (
    df[df["type"] == "export_start"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
starts["start_timestamp"] = pd.to_datetime(starts["timestamp"], utc=True, errors="coerce")
starts["run"] = starts.groupby("resource_id").cumcount() + 1
starts = starts[["resource_id", "run", "start_timestamp", "resource", "resource_type", "resource_label"]]

ends = (
    df[df["type"] == "export_end"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
ends["end_timestamp"] = pd.to_datetime(ends["timestamp"], utc=True, errors="coerce")
ends["run"] = ends.groupby("resource_id").cumcount() + 1
ends = ends[["resource_id", "run", "end_timestamp"]]

df_merged_refresh = starts.merge(ends, on=["resource_id", "run"])

df_merged_refresh["time_diff_minutes"] = (
    (df_merged_refresh["end_timestamp"] - df_merged_refresh["start_timestamp"])
    .dt.total_seconds() / 60
)


##
Duration Analysis

In [ ]:
gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="total", top_n=10)
gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="average", top_n=10)
gencharts.generate_plt_by_resource_type(df,"export_start", top_n=10)

##
Longest running exports

In [ ]:
df_merged_refresh[['resource', 'start_timestamp','end_timestamp','time_diff_minutes','run']].copy().sort_values(by='time_diff_minutes', ascending=False).head(20)